## Name : Rajguru Chiwate
## Roll no. : UAI05BT12440
## DIV: A

# Amazon Reviews NLP Pipeline

1. **Experiment 1**: Text Preprocessing & Cleaning (Contractions, Tokenization, Stopwords Removal, Lemmatization)
2. **Experiment 2**: Feature Extraction using Bag of Words (BoW) & TF-IDF Vectorization
3. **Experiment 3**: Sentiment Classification using Logistic Regression & Model Evaluation

## Experiment 1: Text Preprocessing & Cleaning
In this experiment, we clean raw Amazon review text to prepare it for machine learning:
- Remove null values and map star ratings to binary sentiment labels
- Expand contractions (e.g., *don't* to *do not*)
- Strip noise (HTML tags, URLs, numbers, punctuation)
- Perform word tokenization, stopword removal, and lemmatization

In [1]:
import pandas as pd
import re
import string
import contractions
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

True

### 1. Load Dataset
We load the Amazon Reviews dataset into a pandas DataFrame and examine the first few records.

In [2]:
df = pd.read_csv('data/Amazon_Reviews.csv', engine='python')
df.head()

,Reviewer Name,Profile Link,Country,Review Count,Review Date,Rating,Review Title,Review Text,Date of Experience
0,Eugene ath,/users/66e8185ff1598352d6b3701a,US,1 review,2024-09-16T13:44:26.000Z,Rated 1 out of 5 stars,A Store That Doesn't Want to Sell Anything,"I registered on the website, tried to order a ...","September 16, 2024"
1,Daniel ohalloran,/users/5d75e460200c1f6a6373648c,GB,9 reviews,2024-09-16T18:26:46.000Z,Rated 1 out of 5 stars,Had multiple orders one turned up and…,Had multiple orders one turned up and driver h...,"September 16, 2024"
2,p fisher,/users/546cfcf1000064000197b88f,GB,90 reviews,2024-09-16T21:47:39.000Z,Rated 1 out of 5 stars,I informed these reprobates,I informed these reprobates that I WOULD NOT B...,"September 16, 2024"
3,Greg Dunn,/users/62c35cdbacc0ea0012ccaffa,AU,5 reviews,2024-09-17T07:15:49.000Z,Rated 1 out of 5 stars,Advertise one price then increase it on website,I have bought from Amazon before and no proble...,"September 17, 2024"
4,Sheila Hannah,/users/5ddbe429478d88251550610e,GB,8 reviews,2024-09-16T18:37:17.000Z,Rated 1 out of 5 stars,If I could give a lower rate I would,If I could give a lower rate I would! I cancel...,"September 16, 2024"


Check dataset structure, column data types, and total review counts.

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21214 entries, 0 to 21213
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Reviewer Name       21214 non-null  object
 1   Profile Link        21163 non-null  object
 2   Country             21054 non-null  object
 3   Review Count        21055 non-null  object
 4   Review Date         21055 non-null  object
 5   Rating              21055 non-null  object
 6   Review Title        21055 non-null  object
 7   Review Text         21055 non-null  object
 8   Date of Experience  20947 non-null  object
dtypes: object(9)
memory usage: 1.5+ MB


### 2. Handle Missing Values & Sentiment Labeling
We drop rows with missing reviews or ratings.

For binary sentiment classification:
- **Positive (1)**: 4-star and 5-star reviews
- **Negative (0)**: 1-star and 2-star reviews
- **Neutral (dropped)**: 3-star reviews are excluded to maintain clear binary sentiment separation.

In [4]:
df = df.dropna(subset=['Review Text', 'Rating'])

df = df[df['Rating'] != 'Rated 3 out of 5 stars'].copy()
df['sentiment'] = df['Rating'].map({
    'Rated 1 out of 5 stars': 0,
    'Rated 2 out of 5 stars': 0,
    'Rated 4 out of 5 stars': 1,
    'Rated 5 out of 5 stars': 1
})

df[['Review Text', 'Rating', 'sentiment']].head()

,Review Text,Rating,sentiment
0,"I registered on the website, tried to order a ...",Rated 1 out of 5 stars,0
1,Had multiple orders one turned up and driver h...,Rated 1 out of 5 stars,0
2,I informed these reprobates that I WOULD NOT B...,Rated 1 out of 5 stars,0
3,I have bought from Amazon before and no proble...,Rated 1 out of 5 stars,0
4,If I could give a lower rate I would! I cancel...,Rated 1 out of 5 stars,0


Check the class distribution of positive and negative reviews.

In [5]:
df['sentiment'].value_counts()

sentiment
0    14350
1     5820
Name: count, dtype: int64

### 3. Text Preprocessing Pipeline
We build a text preprocessing function that performs:
1. **Lowercasing**: Uniform lower case conversion.
2. **Contraction Expansion**: Replaces short forms with full words.
3. **Noise Removal**: Strips HTML tags, web URLs, emails, numbers, and repeated characters.
4. **Punctuation Removal**: Removes all punctuation symbols.
5. **Tokenization**: Breaks text into individual tokens.
6. **Stopwords Removal**: Filters out common non-informative words.
7. **Lemmatization**: Converts words to their base root form.

In [6]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    # Convert text to lowercase
    text = str(text).lower()
    
    # Expand contractions (e.g., don't -> do not)
    text = contractions.fix(text)
    
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    
    # Remove website URLs
    text = re.sub(r'https?://\S+|www\S+', '', text)
    
    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    
    # Remove numbers and digits
    text = re.sub(r'\d+', '', text)
    
    # Remove punctuation marks
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # Remove repeated characters (e.g., soooo -> so)
    text = re.sub(r'(.)\1{2,}', r'\1', text)
    
    # Tokenize sentence into words
    tokens = word_tokenize(text)
    
    # Remove stopwords and lemmatize words to root form
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    
    # Join cleaned tokens back into a string
    return ' '.join(tokens)

Apply the preprocessing function to all reviews and preview cleaned text.

In [7]:
df['clean_review'] = df['Review Text'].apply(preprocess_text)
df[['Review Text', 'clean_review']].head()

,Review Text,clean_review
0,"I registered on the website, tried to order a ...",registered website tried order laptop entered ...
1,Had multiple orders one turned up and driver h...,multiple order one turned driver phone door nu...
2,I informed these reprobates that I WOULD NOT B...,informed reprobate would going visit sick rela...
3,I have bought from Amazon before and no proble...,bought amazon problem happy service price amaz...
4,If I could give a lower rate I would! I cancel...,could give lower rate would cancelled amazon p...


Save the preprocessed reviews to a CSV file for reuse in subsequent experiments.

In [8]:
df.to_csv('data/Amazon_Cleaned.csv', index=False)

## Experiment 2: Feature Extraction (BoW & TF-IDF)
In this experiment, we convert the cleaned text into numerical vectors using two feature extraction techniques:
- **Bag of Words (BoW)**: Counts word occurrences.
- **TF-IDF**: Measures relative word importance across documents.

In [9]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import pickle

### 1. Bag of Words (BoW)
Bag of Words creates a matrix where each column represents a word from the vocabulary and values represent word frequencies. We limit the vocabulary to the top 5,000 most frequent words.

In [10]:
bow_vectorizer = CountVectorizer(max_features=5000)
X_bow = bow_vectorizer.fit_transform(df['clean_review'])
X_bow

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 622431 stored elements and shape (20170, 5000)>

Display vocabulary terms extracted by CountVectorizer.

In [11]:
bow_vectorizer.get_feature_names_out()

array(['abandoned', 'ability', 'able', ..., '𝐂𝐎𝐌𝐏𝐋𝐀𝐈𝐍𝐓𝐒𝐒𝐮𝐩𝐩𝐨𝐫𝐭amazon',
       '𝔸𝕞𝕒𝕫𝕠𝕟𝕕𝕚𝕒𝕝', '𝕟𝕦𝕞𝕓𝕖𝕣'], shape=(5000,), dtype=object)

Preview the Bag of Words representation as a DataFrame for the first 5 reviews.

In [12]:
pd.DataFrame(X_bow[:5].toarray(), columns=bow_vectorizer.get_feature_names_out())

,abandoned,ability,able,abroad,abrupt,abruptly,absolute,absolutely,absurd,abuse,...,yous,youtube,yr,zero,zip,zone,𝐂𝐎𝐌𝐏𝐋𝐀𝐈𝐍𝐓𝐒𝐒𝐮𝐩𝐩𝐨𝐫𝐭,𝐂𝐎𝐌𝐏𝐋𝐀𝐈𝐍𝐓𝐒𝐒𝐮𝐩𝐩𝐨𝐫𝐭amazon,𝔸𝕞𝕒𝕫𝕠𝕟𝕕𝕚𝕒𝕝,𝕟𝕦𝕞𝕓𝕖𝕣
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### 2. TF-IDF Vectorization
TF-IDF balances term frequency with document frequency, assigning higher weights to informative words that are specific to individual reviews rather than common across all reviews.

In [13]:
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf_vectorizer.fit_transform(df['clean_review'])
X_tfidf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 622431 stored elements and shape (20170, 5000)>

Display vocabulary terms extracted by TfidfVectorizer.

In [14]:
tfidf_vectorizer.get_feature_names_out()

array(['abandoned', 'ability', 'able', ..., '𝐂𝐎𝐌𝐏𝐋𝐀𝐈𝐍𝐓𝐒𝐒𝐮𝐩𝐩𝐨𝐫𝐭amazon',
       '𝔸𝕞𝕒𝕫𝕠𝕟𝕕𝕚𝕒𝕝', '𝕟𝕦𝕞𝕓𝕖𝕣'], shape=(5000,), dtype=object)

Preview the TF-IDF feature matrix values for the first 5 reviews.

In [15]:
pd.DataFrame(X_tfidf[:5].toarray(), columns=tfidf_vectorizer.get_feature_names_out())

,abandoned,ability,able,abroad,abrupt,abruptly,absolute,absolutely,absurd,abuse,...,yous,youtube,yr,zero,zip,zone,𝐂𝐎𝐌𝐏𝐋𝐀𝐈𝐍𝐓𝐒𝐒𝐮𝐩𝐩𝐨𝐫𝐭,𝐂𝐎𝐌𝐏𝐋𝐀𝐈𝐍𝐓𝐒𝐒𝐮𝐩𝐩𝐨𝐫𝐭amazon,𝔸𝕞𝕒𝕫𝕠𝕟𝕕𝕚𝕒𝕝,𝕟𝕦𝕞𝕓𝕖𝕣
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### 3. Save Vectorizers
Serialize and save both vectorizers to the `assets` directory for future inference.

In [16]:
with open('assets/amazon_bow_vectorizer.pkl', 'wb') as file:
    pickle.dump(bow_vectorizer, file)

with open('assets/amazon_tfidf_vectorizer.pkl', 'wb') as file:
    pickle.dump(tfidf_vectorizer, file)

print('Vectorizers saved successfully.')

Vectorizers saved successfully.


## Experiment 3: Sentiment Classification using Logistic Regression
In this experiment, we train a Logistic Regression classifier on TF-IDF features to predict positive or negative sentiment, and evaluate model performance.

In [17]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

### 1. Train-Test Split
Split the dataset into 80% training data and 20% test data to validate model generalization.

In [18]:
X = X_tfidf
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

### 2. Model Training
Fit the Logistic Regression model using the training features and labels.

In [19]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


### 3. Model Evaluation
Generate predictions on test data and evaluate using Accuracy, Confusion Matrix, and Classification Report.

In [20]:
y_pred = model.predict(X_test)

In [21]:
print('Accuracy:', accuracy_score(y_test, y_pred))
print('\nConfusion Matrix:\n', confusion_matrix(y_test, y_pred))
print('\nClassification Report:\n', classification_report(y_test, y_pred))

Accuracy: 0.9308378780366882

Confusion Matrix:
 [[2786   84]
 [ 195  969]]

Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.97      0.95      2870
           1       0.92      0.83      0.87      1164

    accuracy                           0.93      4034
   macro avg       0.93      0.90      0.91      4034
weighted avg       0.93      0.93      0.93      4034



### 4. Save Model & Sample Predictions
Save the trained classifier to the `Models` directory and test inference on sample review sentences.

In [22]:
with open('Models/amazon_logistic_reg_model.pkl', 'wb') as file:
    pickle.dump(model, file)

print('Model saved successfully.')

Model saved successfully.


In [23]:
new_reviews = [
    'The product arrived quickly and works perfectly, very satisfied!',
    'Defective item, late delivery, and customer service refused to help.'
]

new_features = tfidf_vectorizer.transform(new_reviews)
predictions = model.predict(new_features)

for review, pred in zip(new_reviews, predictions):
    sentiment = 'Positive' if pred == 1 else 'Negative'
    print(f'Review: {review}\nPrediction: {sentiment}\n')

Review: The product arrived quickly and works perfectly, very satisfied!
Prediction: Positive

Review: Defective item, late delivery, and customer service refused to help.
Prediction: Negative

